# 05 · Connecting via Spark Connect (Case B)

**Theory**: docs/05-pyspark-in-practice.md

**Prerequisite**: `make up-cluster` (Spark Standalone: 1 master + 2 workers +
a Spark Connect server, all in Docker).

Your notebook process here is a **thin gRPC client** — no JVM, no Hadoop
classpath, nothing heavy. All compute happens inside the `spark-connect`
container. That also means: **file paths you reference must exist inside the
containers**, not on your laptop — that's exactly why `docker-compose.yml`
bind-mounts `./data` at `/data` in every Spark service.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import layer_path
from pyspark.sql import SparkSession

# Cases B/D: thin client over Spark Connect (make up-cluster started the
# server). All heavy compute happens in the Docker cluster — this process
# only sends the unresolved logical plan over gRPC and gets results back.
# Remember: any file path you reference (e.g. "/data/bronze/vendas") is
# resolved *inside* the containers, not on your laptop — that is exactly why
# docker-compose.yml mounts ./data at /data in every Spark container.
spark = (
    SparkSession.builder.appName("05-spark-connect")
    .remote("sc://localhost:15002")
    .getOrCreate()
)
spark

## Proving execution really happens in the cluster

`spark.range(...).count()` is cheap enough to run anywhere — the interesting
check is confirming the *Spark UI* you see is the cluster's, not something
local. Open http://localhost:4040 (the Connect server's driver UI) and
http://localhost:8080 (the Standalone Master UI) in your browser while
running the next cell, and watch the job appear.

In [ ]:
df = spark.range(0, 20_000_000)
print(f"Row count: {df.count():,}")
print("Check http://localhost:8080 -> Running Applications, and")
print("      http://localhost:4040 -> Jobs tab, to see this execution.")

## Reading the shared dataset

Remember: `/data/...` is the path *inside* the containers (see
`docker-compose.yml`'s `x-spark-common` volume mount). Run
`make generate-data SCALE=large` on the host first — since `./data` is bind
-mounted, the containers see it immediately, no copy step needed.

In [ ]:
vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))
categorias = spark.read.parquet(layer_path("connect", "bronze", "categorias"))

print(f"vendas: {vendas.count():,} rows")
categorias.show(5)

## Partition tour

Spark Connect's Python client intentionally does **not** expose the low
-level RDD API (`df.rdd`) — it's a DataFrame/SQL-only client. To inspect
partitions, use `spark_partition_id()` as a regular column instead.

In [ ]:
from pyspark.sql.functions import spark_partition_id

partition_counts = (
    vendas.groupBy(spark_partition_id().alias("partition_id"))
    .count()
    .orderBy("partition_id")
)
partition_counts.show(20)
print(f"Total partitions: {partition_counts.count()}")

In [ ]:
spark.stop()